# HealthAI Disease Classifier — Synthetic Dataset Edition
## Bio_ClinicalBERT · 12 diseases · 7200 balanced samples

**Training data:** `healthai_synthetic_training_set_7200.csv` / `.jsonl`  
**No Supabase required** — all data loaded from uploaded files.

**All bugs fixed:**
- Lab context placed first (never truncated)
- Augmentation separator consistent between train and inference
- Class-weighted loss + label smoothing
- Early stopping on validation loss
- GPU guard — fails fast if no GPU
- `num_workers` safe for Windows and Colab

**Before running:** Runtime → Change runtime type → T4 GPU  
**Upload files:** Ensure both `healthai_synthetic_training_set_7200.csv` and `.jsonl` are in the Colab file browser.


## Step 1 — Install libraries


In [ ]:
!pip install transformers torch scikit-learn matplotlib seaborn --quiet
print('All libraries ready.')


## Step 2 — Configuration


In [ ]:
# ── Paths — adjust if files are in a subfolder ───────────────────────────
CSV_PATH  = 'healthai_synthetic_training_set_7200.csv'
JSONL_PATH = 'healthai_synthetic_training_set_7200.jsonl'

# ── Model & training hyperparameters ──────────────────────────────────────
MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_LEN    = 128   # synthetic texts avg 139 chars ≈ 30 tokens; 128 is more than enough
BATCH_SIZE = 32    # larger batch safe with short texts
EPOCHS     = 6
LR         = 2e-5
PATIENCE   = 2     # early stopping
SEED       = 42

DISEASE_LABELS = {
    'epilepsy':0,'migraine':1,'stroke':2,'diabetic_neuropathy':3,
    'dementia':4,'parkinsons':5,'dengue':6,'malaria':7,
    'kala_azar':8,'chikungunya':9,'japanese_encephalitis':10,'scrub_typhus':11,
}
LABEL_NAMES = {v: k for k, v in DISEASE_LABELS.items()}
NUM_LABELS  = len(DISEASE_LABELS)
print(f'Config ready. {NUM_LABELS} disease classes.')


## Step 3 — Load & validate dataset


In [ ]:
import pandas as pd, json, random, os
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

# Load from CSV (primary); fall back to JSONL if CSV not found
if os.path.exists(CSV_PATH):
    df_raw = pd.read_csv(CSV_PATH)
    print(f'Loaded CSV: {len(df_raw)} rows')
elif os.path.exists(JSONL_PATH):
    rows = [json.loads(l) for l in open(JSONL_PATH)]
    df_raw = pd.DataFrame(rows)
    print(f'Loaded JSONL: {len(df_raw)} rows')
else:
    raise FileNotFoundError(
        f'Neither {CSV_PATH} nor {JSONL_PATH} found.\n'
        'Upload the files to the Colab file browser and re-run.'
    )

# Validate columns
assert 'label' in df_raw.columns and 'input_text' in df_raw.columns, \
    'Expected columns: label, input_text'

# Drop unknowns, map to int labels
df_raw = df_raw[df_raw['label'].isin(DISEASE_LABELS)].copy()
df_raw['label_id'] = df_raw['label'].map(DISEASE_LABELS)
df_raw['input_text'] = df_raw['input_text'].astype(str).str.strip()

print(f'Valid samples: {len(df_raw)}')
print()
print('Samples per disease:')
print(df_raw['label'].value_counts().to_string())
print()
print(f'Text length — mean: {df_raw["input_text"].str.len().mean():.0f} chars  '
      f'max: {df_raw["input_text"].str.len().max()} chars')


## Step 4 — Train / validation / test split


In [ ]:
from sklearn.model_selection import train_test_split

# 70% train | 15% val | 15% test  — stratified
train_df, temp_df = train_test_split(
    df_raw, test_size=0.30, random_state=SEED, stratify=df_raw['label_id'])
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label_id'])

print(f'Train : {len(train_df):5d} samples')
print(f'Val   : {len(val_df):5d} samples')
print(f'Test  : {len(test_df):5d} samples')
print()
print('Train label distribution:')
print(train_df['label'].value_counts().to_string())


## Step 5 — GPU check & model load


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if str(DEVICE) != 'cuda':
    raise RuntimeError(
        'GPU not detected. Training will take hours on CPU.\n'
        'Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU, then re-run.'
    )
print(f'GPU: {torch.cuda.get_device_name(0)}')

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
model     = model.to(DEVICE)
print(f'Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')


## Step 6 — Tokenize & build DataLoaders


In [ ]:
from torch.utils.data import Dataset, DataLoader
import platform

class DiseaseDataset(Dataset):
    def __init__(self, texts, labels, tok, max_len):
        self.texts  = list(texts)
        self.labels = list(labels)
        self.tok    = tok
        self.max_len= max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(
            str(self.texts[i]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[i], dtype=torch.long)
        }

# num_workers: 0 on Windows (avoids freeze), 2 on Colab/Linux
_nw = 0 if platform.system() == 'Windows' else 2

train_ds = DiseaseDataset(train_df['input_text'], train_df['label_id'], tokenizer, MAX_LEN)
val_ds   = DiseaseDataset(val_df['input_text'],   val_df['label_id'],   tokenizer, MAX_LEN)
test_ds  = DiseaseDataset(test_df['input_text'],  test_df['label_id'],  tokenizer, MAX_LEN)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=_nw, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=_nw, pin_memory=True)

print(f'Train batches: {len(train_dl)}  |  Val batches: {len(val_dl)}  |  Test batches: {len(test_dl)}')
print(f'MAX_LEN={MAX_LEN}, BATCH_SIZE={BATCH_SIZE}, num_workers={_nw}')


## Step 7 — Fine-tune with class-weighted loss + early stopping


In [ ]:
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup

# Class weights — perfectly balanced here but good practice
label_counts  = train_df['label_id'].value_counts().sort_index()
class_weights = torch.tensor(
    [1.0 / label_counts[i] for i in range(NUM_LABELS)], dtype=torch.float
).to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_LABELS
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(0.06 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

train_losses, train_accs, val_losses, val_accs = [], [], [], []
best_val_loss = float('inf')
patience_ctr  = 0
best_epoch    = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0
    for step, batch in enumerate(train_dl):
        optimizer.zero_grad()
        ids   = batch['input_ids'].to(DEVICE)
        mask  = batch['attention_mask'].to(DEVICE)
        lbls  = batch['label'].to(DEVICE)
        out   = model(input_ids=ids, attention_mask=mask)
        loss  = criterion(out.logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        t_loss    += loss.item()
        preds      = torch.argmax(out.logits, dim=1)
        t_correct += (preds == lbls).sum().item()
        t_total   += lbls.size(0)
        if (step + 1) % 40 == 0:
            print(f'  Ep {epoch} step {step+1}/{len(train_dl)} '
                  f'loss={t_loss/(step+1):.4f} acc={t_correct/t_total*100:.1f}%')
    ep_t_loss = t_loss / len(train_dl)
    ep_t_acc  = t_correct / t_total * 100
    train_losses.append(ep_t_loss)
    train_accs.append(ep_t_acc)

    # ── Validate ────────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_dl:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            lbls = batch['label'].to(DEVICE)
            out  = model(input_ids=ids, attention_mask=mask)
            v_loss    += criterion(out.logits, lbls).item()
            preds      = torch.argmax(out.logits, dim=1)
            v_correct += (preds == lbls).sum().item()
            v_total   += lbls.size(0)
    ep_v_loss = v_loss / len(val_dl)
    ep_v_acc  = v_correct / v_total * 100
    val_losses.append(ep_v_loss)
    val_accs.append(ep_v_acc)

    print(f'Epoch {epoch}/{EPOCHS} '
          f'| Train loss={ep_t_loss:.4f} acc={ep_t_acc:.2f}% '
          f'| Val loss={ep_v_loss:.4f} acc={ep_v_acc:.2f}%')

    # ── Early stopping ──────────────────────────────────────────────────────
    if ep_v_loss < best_val_loss:
        best_val_loss = ep_v_loss
        best_epoch    = epoch
        patience_ctr  = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'  ✓ Best model saved (val_loss={ep_v_loss:.4f})')
    else:
        patience_ctr += 1
        print(f'  No improvement ({patience_ctr}/{PATIENCE})')
        if patience_ctr >= PATIENCE:
            print(f'Early stopping triggered at epoch {epoch}.')
            break

model.load_state_dict(torch.load('best_model.pt'))
print(f'\nBest model from epoch {best_epoch} restored. Training complete.')


## Step 8 — Evaluate on held-out test set


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np

model.eval()
all_preds, all_labels_list = [], []
with torch.no_grad():
    for batch in test_dl:
        out = model(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE)
        )
        all_preds.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        all_labels_list.extend(batch['label'].numpy())

acc = accuracy_score(all_labels_list, all_preds)
f1m = f1_score(all_labels_list, all_preds, average='macro')
f1w = f1_score(all_labels_list, all_preds, average='weighted')
label_list = [LABEL_NAMES[i] for i in range(NUM_LABELS)]

print('=' * 58)
print('HealthAI — Test Set Results')
print('=' * 58)
print(f'Accuracy (overall)  : {acc*100:.2f}%')
print(f'F1 macro            : {f1m:.4f}')
print(f'F1 weighted         : {f1w:.4f}')
print()
print(classification_report(all_labels_list, all_preds, target_names=label_list))


## Step 9 — Confusion matrix & training curves


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt

# Confusion matrix
cm = confusion_matrix(all_labels_list, all_preds)
plt.figure(figsize=(14, 11))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=label_list, yticklabels=label_list, cmap='Blues')
plt.title('HealthAI — Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Training curves
epochs_ran = len(train_losses)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(range(1, epochs_ran+1), train_losses, 'o-', color='firebrick',  label='Train', linewidth=2)
axes[0].plot(range(1, epochs_ran+1), val_losses,   's--', color='steelblue', label='Val',   linewidth=2)
axes[0].set_title('Loss', fontweight='bold'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, epochs_ran+1), train_accs, 'o-', color='firebrick',  label='Train', linewidth=2)
axes[1].plot(range(1, epochs_ran+1), val_accs,   's--', color='steelblue', label='Val',   linewidth=2)
axes[1].set_title('Accuracy (%)', fontweight='bold'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('HealthAI Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Charts saved.')


## Step 10 — Live prediction demo


In [ ]:
def predict(text):
    """Predict disease from a clinical text string."""
    model.eval()
    enc = tokenizer(
        str(text),
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    with torch.no_grad():
        out = model(
            input_ids=enc['input_ids'].to(DEVICE),
            attention_mask=enc['attention_mask'].to(DEVICE)
        )
    probs = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
    pred  = int(probs.argmax())
    top3  = [(LABEL_NAMES[int(i)], round(float(probs[i]) * 100, 1))
             for i in probs.argsort()[-3:][::-1]]
    conf  = round(float(probs[pred]) * 100, 1)
    if conf < 40:
        print(f'  [LOW CONFIDENCE: {conf}% — consider clinical review]')
    return LABEL_NAMES[pred], conf, top3


test_cases = [
    ('High fever 39C, severe headache, retro-orbital pain, rash, myalgia. '
     'NS1 antigen positive. Platelet count 42000. Leucopenia WBC 3200.',          'dengue'),
    ('Severe unilateral throbbing headache, photophobia, phonophobia, nausea. '
     'NS1 antigen NEGATIVE. Platelet count 210000. No fever. Normal CBC.',        'migraine'),
    ('High fever 40C, rigor, sweating, headache. '
     'Blood smear P. vivax trophozoites. RDT pLDH positive.',                    'malaria'),
    ('High fever 3 weeks, eschar on axilla, lymphadenopathy. '
     'Weil-Felix OXK titre 1:160. IgM ELISA scrub typhus positive.',             'scrub_typhus'),
    ('Prolonged fever 4 weeks, massive splenomegaly, weight loss, Bihar. '
     'rK39 rapid test strongly positive. Haemoglobin 7.2.',                      'kala_azar'),
    ('Breakthrough seizures, EEG generalised spike-wave, on levetiracetam. '
     'EEG 3Hz spike-wave. AED levels subtherapeutic.',                           'epilepsy'),
    ('Progressive memory loss, confusion, wandering, age 74. '
     'MMSE 18. MoCA 16. MRI hippocampal atrophy bilateral.',                     'dementia'),
    ('Sudden right-sided weakness, aphasia, onset 2 hours ago. '
     'CT no haemorrhage. MRI DWI left MCA acute infarct.',                       'stroke'),
    ('Resting tremor right hand, bradykinesia, rigidity, shuffling gait. '
     'DaT scan reduced uptake. Levodopa response positive.',                     'parkinsons'),
    ('Burning feet, numbness, reduced vibration sense, HbA1c 9.2%. '
     'Nerve conduction velocity reduced. 10g monofilament positive.',             'diabetic_neuropathy'),
    ('High fever, polyarthralgia, bilateral joint swelling after mosquito bite. '
     'IgM ELISA chikungunya positive. Lymphopenia.',                             'chikungunya'),
    ('Fever, altered consciousness, neck stiffness, post-monsoon. '
     'CSF IgM JE positive. MRI bilateral thalamic T2 hyperintensities.',         'japanese_encephalitis'),
]

print('HealthAI — Live Predictions')
print('=' * 68)
correct = 0
for text, expected in test_cases:
    disease, conf, top3 = predict(text)
    tick = 'CORRECT' if disease == expected else 'WRONG'
    if disease == expected: correct += 1
    print(f'Input    : {text[:72]}...')
    print(f'Predicted: {disease.upper()} ({conf}%) [{tick}]  Expected: {expected}')
    print(f'Top 3    : {top3}')
    print()
print(f'Demo accuracy: {correct}/{len(test_cases)} correct ({correct/len(test_cases)*100:.0f}%)')


## Step 11 — Save model & package for download


In [ ]:
import os, zipfile, json as json_lib
from datetime import datetime

SAVE_DIR = 'healthai_classifier_synthetic'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json_lib.dump(LABEL_NAMES, f, indent=2)

with open(f'{SAVE_DIR}/model_info.json', 'w') as f:
    json_lib.dump({
        'version':       'synthetic-v1',
        'base_model':    MODEL_NAME,
        'training_data': 'healthai_synthetic_training_set_7200',
        'train_samples': len(train_df),
        'val_samples':   len(val_df),
        'test_samples':  len(test_df),
        'accuracy':      round(acc * 100, 2),
        'f1_macro':      round(f1m, 4),
        'f1_weighted':   round(f1w, 4),
        'best_epoch':    best_epoch,
        'diseases':      list(DISEASE_LABELS.keys()),
        'trained':       datetime.now().isoformat(),
    }, f, indent=2)

ZIP = 'healthai_model_synthetic.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(SAVE_DIR):
        zf.write(f'{SAVE_DIR}/{fn}', fn)
    for fn in ['confusion_matrix.png', 'training_curves.png']:
        if os.path.exists(fn):
            zf.write(fn, fn)

size_mb = os.path.getsize(ZIP) / 1e6
print(f'Saved: {ZIP}  ({size_mb:.1f} MB)')
print('Download: Files panel → right-click healthai_model_synthetic.zip → Download')
print()
print('=' * 55)
print('TRAINING COMPLETE')
print('=' * 55)
print(f'Accuracy : {acc*100:.2f}%')
print(f'F1 macro : {f1m:.4f}')
print(f'Best epoch: {best_epoch}')
print('=' * 55)
